In [1]:
import pickle  
import os 
from tqdm import tqdm
from langchain_ollama import OllamaEmbeddings
from itext2kg.models import KnowledgeGraph, Entity, Relationship
from itext2kg.models.knowledge_graph import EntityProperties

/home/mindrank/miniconda3/envs/kg/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [5]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest", base_url="http://localhost:11434")

In [6]:
txt_dir = "/home/mindrank/fuli/itext2kg/Data/AD_pubtabor"
kg_dir = "/home/mindrank/fuli/itext2kg/output_kg/AD"
sub_kg_list = os.listdir(kg_dir)

In [7]:
de_text_list = [i.split(".")[0] for i in os.listdir(txt_dir)]
de_kg_list = [i.split(".")[0] for i in os.listdir(kg_dir)]
de_intersection = list(set(de_text_list) & set(de_kg_list))
len(de_intersection)

279596

In [5]:
# # 进行后处理
# for pmid in tqdm(sub_kg_list, desc="Processing PMIDs", unit="file"):
#     # if os.path.exists(f"/home/mindrank/fuli/itext2kg/output_kg/all_kg_0407/{pmid}"):
#     #     pass
#     # else:
#     try:
#         with open(f"{kg_dir}/{pmid}", 'rb') as f:
#             kg = pickle.load(f)
        
#         if os.path.exists(f"{txt_dir}/{pmid.replace('pkl', 'txt')}"):
#             with open(f"{txt_dir}/{pmid.replace('pkl', 'txt')}", "r") as f: #self.variable
#                 text_line = f.readlines() # Strip AND filter out empty lines
#                 PMID = text_line[0].strip().split('|')[0]
#                 title = text_line[0].strip().split('|')[-1]
#                 abstract = text_line[1].strip().split('|')[-1]
#                 context = f"Title: {title}Abstract: {abstract}."
            
#             global_entities = []
#             all_entities = []
#             for entity in kg.entities:
#                 all_entities.append(entity.label)
#                 if entity.label == 'abstract':
#                     entity.name = f"pmid{pmid.split('.')[0]}"
#                     entity.properties_info['context'] = context
#                     embeddings_info = embeddings.embed_query(context)
#                     entity.properties.embeddings = embeddings_info
#                 global_entities.append(entity)
            
#             if 'abstract' not in all_entities:
#                 embeddings_info = embeddings.embed_query(context)
#                 ab_entity = Entity(
#                     label= "abstract", 
#                     name= f"pmid{pmid.split('.')[0]}", 
#                     properties_info={'context': context},
#                     properties = EntityProperties(embeddings=embeddings_info)
#                     )
#                 global_entities.append(ab_entity)

#             curated_relationships = []
#             for re in kg.relationships:
#                 curated_relationships.append(re)
                
#             for entity in global_entities:
#                 if entity.label == "disease":
#                     curated_relationships.append(
#                         Relationship(
#                             startEntity= ab_entity, 
#                             endEntity = entity,
#                             name = "reported")
#                         )
                            
#             new_kg = KnowledgeGraph(
#                     entities=global_entities, 
#                     relationships=curated_relationships,
#                     )
#             with open(f"/home/mindrank/fuli/itext2kg/output_kg/all_kg_0407/{pmid}", 'wb') as f:
#                     pickle.dump(kg, f)
    
#     except:
#         os.system(f"mv {kg_dir}/{pmid} /home/mindrank/fuli/itext2kg/output_kg/AD_failed/")

In [8]:
output_dir = "/home/mindrank/fuli/itext2kg/output_kg/all_kg_0508"
# sub_kg_list = os.listdir(kg_dir) # 假设这里获取了 pkl 文件名列表

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)


# 进行后处理
for pmid_filename in tqdm(de_intersection, desc="Processing PMIDs", unit="file"):
    pmid_filename = f"{pmid_filename}.pkl"
    output_path = os.path.join(output_dir, pmid_filename)

    original_kg_path = os.path.join(kg_dir, pmid_filename)
    txt_filename = pmid_filename.replace('.pkl', '.txt')
    txt_path = os.path.join(txt_dir, txt_filename)

    if os.path.exists(output_path):
        # print(f"Output file already exists, skipping: {output_path}")
        continue
    if not os.path.exists(original_kg_path):
        # print(f"Warning: Original KG file not found: {original_kg_path}")
        continue
    if not os.path.exists(txt_path):
        # print(f"Warning: Corresponding TXT file not found: {txt_path}")
        continue
    if os.path.exists(output_path):
        continue

    try:
        # 1. 加载原始 KG
        with open(original_kg_path, 'rb') as f:
            original_kg = pickle.load(f)

        if len(original_kg.entities) > 0:
            # 2. 读取文本文件获取 context
            context = ""
            pmid_str = pmid_filename.split('.')[0] # Extract PMID digits from filename
            title = ""
            abstract = ""
            try:
                with open(txt_path, "r", encoding='utf-8') as f: # Specify encoding
                    lines = f.readlines()
                    if len(lines) >= 1:
                        parts = lines[0].strip().split('|')
                        # Assuming PMID is always first part if present, title last
                        # pmid_from_file = parts[0] # Optional: verify against filename
                        title = parts[-1]
                    if len(lines) >= 2:
                        parts = lines[1].strip().split('|')
                        # Assuming abstract is last part of second line
                        abstract = parts[-1]
                    context = f"Title: {title} Abstract: {abstract}" # Note: removed trailing '.' for consistency
                    if not title and not abstract:
                        print(f"Warning: Could not extract title or abstract from {txt_path}")
                        context = "Context unavailable." # Handle cases where parsing fails

            except IndexError:
                # print(f"Warning: Error parsing TXT file format for {txt_path}")
                context = "Context parsing error." # Handle parsing errors
            except Exception as e:
                # print(f"Error reading TXT file {txt_path}: {e}")
                continue # Skip this file if text can't be read

            # 3. 处理实体，确保 abstract 实体存在并更新
            global_entities = []
            abstract_node = None # 用来存储找到的或新建的 abstract 实体
            found_abstract = False

            
            for entity in original_kg.entities:
                if entity.label == 'abstract':
                    # 找到已存在的 abstract 实体
                    entity.name = f"pmid{pmid_str}"
                    entity.properties_info['context'] = context
                    embeddings_info = embeddings.embed_query(context)
                    # 确保 properties 存在并且是正确的类型
                    if not isinstance(entity.properties, EntityProperties):
                        entity.properties = EntityProperties() # 创建一个如果不存在或类型不对
                    entity.properties.embeddings = embeddings_info
                    abstract_node = entity # 引用找到的实体
                    found_abstract = True
                    # print(f"Updated existing abstract node for {pmid_filename}")
                global_entities.append(entity) # 添加到新列表

            if not found_abstract:
                # 如果原始 KG 中没有 abstract 实体，则创建一个新的
                # print(f"Creating new abstract node for {pmid_filename}")
                embeddings_info = embeddings.embed_query(context)
                abstract_node = Entity(
                    label="abstract",
                    name=f"pmid{pmid_str}",
                    properties_info={'context': context},
                    properties=EntityProperties(embeddings=embeddings_info)
                )
                global_entities.append(abstract_node) # 添加新创建的实体

            # 4. 处理关系
            curated_relationships = []
            # 复制原始关系
            for rel in original_kg.relationships:
                curated_relationships.append(rel)

            # 添加 abstract -> disease 的 "reported" 关系
            if abstract_node: # 确保 abstract_node 确实被定义了
                for entity in global_entities:
                    # Make sure we don't link abstract to itself if it were somehow labeled 'disease'
                    if entity.label == "disease" and entity is not abstract_node:
                        # print(f"Adding 'reported' relationship from abstract to {entity.name}")
                        curated_relationships.append(
                            Relationship(
                                startEntity=abstract_node, # 使用找到的或新建的 abstract_node
                                endEntity=entity,
                                name="reported"
                            )
                        )
            else:
                print(f"Error: Abstract node was not found or created for {pmid_filename}, cannot add relationships.")


            # 5. 创建新的 KG 对象
            new_kg = KnowledgeGraph(
                entities=global_entities,
                relationships=curated_relationships,
            )

            # 6. 保存 *修改后* 的 new_kg 对象
            with open(output_path, 'wb') as f:
                pickle.dump(new_kg, f) # *** 保存 new_kg ***

    except FileNotFoundError:
        print(f"Error: File not found during processing {pmid_filename}. Check paths.")
    except pickle.PickleError as e:
        print(f"Error loading/dumping pickle file {pmid_filename}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred processing {pmid_filename}: {e}")

print("Processing finished.")

Processing PMIDs:  23%|██▎       | 64869/279596 [11:43<32:58, 108.53file/s] 

An unexpected error occurred processing 22272179.pkl: Ran out of input


Processing PMIDs:  42%|████▏     | 116315/279596 [20:58<28:58, 93.93file/s] 

An unexpected error occurred processing 23278303.pkl: Ran out of input


Processing PMIDs:  78%|███████▊  | 217458/279596 [39:01<11:16, 91.79file/s] 

An unexpected error occurred processing 24058519.pkl: Ran out of input


Processing PMIDs: 100%|██████████| 279596/279596 [50:17<00:00, 92.64file/s] 

Processing finished.


In [7]:
txt_dir = "/home/mindrank/fuli/itext2kg/Data/delirium"
kg_dir = "/home/mindrank/fuli/itext2kg/output_kg/delirium"
sub_kg_list = os.listdir(kg_dir)

de_text_list = [i.split(".")[0] for i in os.listdir(txt_dir)]
de_kg_list = [i.split(".")[0] for i in os.listdir(kg_dir)]
de_intersection = list(set(de_text_list) & set(de_kg_list))
print(len(de_intersection))

output_dir = "/home/mindrank/fuli/itext2kg/output_kg/all_kg_0407"
# sub_kg_list = os.listdir(kg_dir) # 假设这里获取了 pkl 文件名列表

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)


# 进行后处理
# for pmid_filename in tqdm(de_intersection, desc="Processing PMIDs", unit="file"):
#     pmid_filename = f"{pmid_filename}.pkl"
#     output_path = os.path.join(output_dir, pmid_filename)

#     original_kg_path = os.path.join(kg_dir, pmid_filename)
#     txt_filename = pmid_filename.replace('.pkl', '.txt')
#     txt_path = os.path.join(txt_dir, txt_filename)

#     if not os.path.exists(original_kg_path):
#         print(f"Warning: Original KG file not found: {original_kg_path}")
#         continue
#     if not os.path.exists(txt_path):
#         print(f"Warning: Corresponding TXT file not found: {txt_path}")
#         continue
#     # if os.path.exists(output_path):
#     #     continue

#     try:
#         # 1. 加载原始 KG
#         with open(original_kg_path, 'rb') as f:
#             original_kg = pickle.load(f)

#         if len(original_kg.entities) > 0:
#             # 2. 读取文本文件获取 context
#             context = ""
#             pmid_str = pmid_filename.split('.')[0] # Extract PMID digits from filename
#             title = ""
#             abstract = ""
#             try:
#                 with open(txt_path, "r", encoding='utf-8') as f: # Specify encoding
#                     lines = f.readlines()
#                     if len(lines) >= 1:
#                         parts = lines[0].strip().split('|')
#                         # Assuming PMID is always first part if present, title last
#                         # pmid_from_file = parts[0] # Optional: verify against filename
#                         title = parts[-1]
#                     if len(lines) >= 2:
#                         parts = lines[1].strip().split('|')
#                         # Assuming abstract is last part of second line
#                         abstract = parts[-1]
#                     context = f"Title: {title} Abstract: {abstract}" # Note: removed trailing '.' for consistency
#                     if not title and not abstract:
#                         print(f"Warning: Could not extract title or abstract from {txt_path}")
#                         context = "Context unavailable." # Handle cases where parsing fails

#             except IndexError:
#                 print(f"Warning: Error parsing TXT file format for {txt_path}")
#                 context = "Context parsing error." # Handle parsing errors
#             except Exception as e:
#                 print(f"Error reading TXT file {txt_path}: {e}")
#                 continue # Skip this file if text can't be read

#             # 3. 处理实体，确保 abstract 实体存在并更新
#             global_entities = []
#             abstract_node = None # 用来存储找到的或新建的 abstract 实体
#             found_abstract = False

            
#             for entity in original_kg.entities:
#                 if entity.label == 'abstract':
#                     # 找到已存在的 abstract 实体
#                     entity.name = f"pmid{pmid_str}"
#                     entity.properties_info['context'] = context
#                     embeddings_info = embeddings.embed_query(context)
#                     # 确保 properties 存在并且是正确的类型
#                     if not isinstance(entity.properties, EntityProperties):
#                         entity.properties = EntityProperties() # 创建一个如果不存在或类型不对
#                     entity.properties.embeddings = embeddings_info
#                     abstract_node = entity # 引用找到的实体
#                     found_abstract = True
#                     # print(f"Updated existing abstract node for {pmid_filename}")
#                 global_entities.append(entity) # 添加到新列表

#             if not found_abstract:
#                 # 如果原始 KG 中没有 abstract 实体，则创建一个新的
#                 # print(f"Creating new abstract node for {pmid_filename}")
#                 embeddings_info = embeddings.embed_query(context)
#                 abstract_node = Entity(
#                     label="abstract",
#                     name=f"pmid{pmid_str}",
#                     properties_info={'context': context},
#                     properties=EntityProperties(embeddings=embeddings_info)
#                 )
#                 global_entities.append(abstract_node) # 添加新创建的实体

#             # 4. 处理关系
#             curated_relationships = []
#             # 复制原始关系
#             for rel in original_kg.relationships:
#                 curated_relationships.append(rel)

#             # 添加 abstract -> disease 的 "reported" 关系
#             if abstract_node: # 确保 abstract_node 确实被定义了
#                 for entity in global_entities:
#                     # Make sure we don't link abstract to itself if it were somehow labeled 'disease'
#                     if entity.label == "disease" and entity is not abstract_node:
#                         # print(f"Adding 'reported' relationship from abstract to {entity.name}")
#                         curated_relationships.append(
#                             Relationship(
#                                 startEntity=abstract_node, # 使用找到的或新建的 abstract_node
#                                 endEntity=entity,
#                                 name="reported"
#                             )
#                         )
#             else:
#                 print(f"Error: Abstract node was not found or created for {pmid_filename}, cannot add relationships.")


#             # 5. 创建新的 KG 对象
#             new_kg = KnowledgeGraph(
#                 entities=global_entities,
#                 relationships=curated_relationships,
#             )

#             # 6. 保存 *修改后* 的 new_kg 对象
#             with open(output_path, 'wb') as f:
#                 pickle.dump(new_kg, f) # *** 保存 new_kg ***

#     except FileNotFoundError:
#         print(f"Error: File not found during processing {pmid_filename}. Check paths.")
#     except pickle.PickleError as e:
#         print(f"Error loading/dumping pickle file {pmid_filename}: {e}")
#     except Exception as e:
#         print(f"An unexpected error occurred processing {pmid_filename}: {e}")

# print("Processing finished.")

44885


In [8]:
"36439572" in de_intersection

True